# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

No longer used.

In [1]:
import polars as pl
import glob
import os
import screed
import csv
import screed

In [3]:
seq_to_species_df = (
    pl.read_csv('../outputs.mapping/cds-min50-singleclust/species_to_genes.csv')
    .rename({ 'ident': 'gene' })
    .select(['gene', 'species'])
)
seq_to_species_df

gene,species
str,str
"""AtH2023_ERR1135319_MAG02_1_7""","""s__Bariatricus sp004560705"""
"""AtH2023_ERR1135319_MAG02_1_10""","""s__Bariatricus sp004560705"""
"""AtH2023_ERR1135319_MAG02_1_13""","""s__Bariatricus sp004560705"""
"""AtH2023_ERR1135319_MAG02_1_23""","""s__Bariatricus sp004560705"""
"""AtH2023_ERR1135319_MAG02_1_27""","""s__Bariatricus sp004560705"""
…,…
"""AtH2023_SRR8960831_MAG08_7_11""","""s__UBA2868 sp004552595"""
"""AtH2023_ERR1135443_MAG08_149_1""","""s__UBA2868 sp004552595"""
"""AtH2023_SRR14369154_MAG04_52_1…","""s__UBA2868 sp004552595"""


In [11]:
seq_to_species_df.group_by('species').agg(pl.len()).sort('len')

species,len
str,u32
"""s__JAFBIX01 sp021531895""",1
"""s__Floccifex porci""",18
"""s__Mogibacterium_A kristiansen…",19
"""s__Fimisoma sp002320005""",39
"""s__Lactobacillus amylovorus""",130
…,…
"""s__UBA2868 sp004552595""",683
"""s__Cryptobacteroides sp9005469…",696
"""s__Ornithospirochaeta sp022785…",750


## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [12]:
DEPTH_PATTERN = '../outputs.mapping/bams.cds3.min50.rand/{m}.x.all-dedup-95.depth.txt'
METAG='SRR11183784'
exclude_ends = 75

def read_depth_txt(metag, seq_to_species_df):
    df = (pl.scan_csv(DEPTH_PATTERN.format(m=metag), include_file_paths="filename",
                       separator='\t', has_header=False,
                       new_columns=('gene', 'pos', 'cov', 'foo'))
          .select(['gene', 'pos', 'cov'])
         ).collect()
    
    xx_df = df.group_by('gene').all().with_columns(
            # select slice [75:-75]
            (pl.col("pos").list.slice(exclude_ends, -exclude_ends).list.len()).alias("len"),
            (pl.col("cov").list.slice(exclude_ends, -exclude_ends)),
            (pl.col("cov").list.slice(exclude_ends, -exclude_ends).list.filter(pl.element() > 0)).list.len().alias("hits"),
        ).with_columns(
            (pl.lit(metag).alias("metag")),
    #        (pl.lit(species).alias("species")),
            # summarize: average depth across contig,
            (pl.col("cov").list.sum() / pl.col("len")).alias("depth_all"),
            # average depth across covered bases,
            (pl.col("cov").list.sum() / pl.col("hits")).alias("depth_cov"),
            # fraction of bases covered
            (pl.col("hits") / pl.col("cov").list.len()).alias("breadth"),
        ).select(["metag", "gene", "len", "hits", "breadth", "depth_all", "depth_cov"])
    
    
    xx_df = xx_df.join(seq_to_species_df, on='gene', how='inner')
    return xx_df

sum_df = read_depth_txt(METAG, seq_to_species_df)
sum_df

Overwriting xx.py


In [5]:
# Read them all in!
rand100_names = [ x.strip() for x in open('../inputs.mapping/rand_subset.3216.100.txt') ]
print(len(rand100_names))


dflist = []
for i, metag in enumerate(rand100_names):
    if i % 10 == 0:
        print(f"{i} of {len(rand100_names)}")
    dflist.append(read_depth_txt(metag, seq_to_species_df))

depth_df = pl.concat(dflist)

print(f"read {len(rand100_names)} depth files.")

100
0 of 100
10 of 100
20 of 100
30 of 100
40 of 100
50 of 100
60 of 100
70 of 100
80 of 100
90 of 100
read 100 depth files.


In [8]:
depth_df['metag'].n_unique()

100

In [9]:
depth_df['gene'].n_unique()

7176

In [7]:
depth_df

metag,gene,len,hits,breadth,depth_all,depth_cov,species
str,str,u32,u32,f64,f64,f64,str
"""ERR3211766""","""AtH2023_ERR1135319_MAG02_1_7""",1572,919,0.584606,1.110051,1.898803,"""s__Bariatricus sp004560705"""
"""ERR3211766""","""AtH2023_ERR1135319_MAG02_1_10""",2022,1031,0.509891,0.654303,1.28322,"""s__Bariatricus sp004560705"""
"""ERR3211766""","""AtH2023_ERR1135319_MAG02_1_13""",861,0,0.0,0.0,NaN,"""s__Bariatricus sp004560705"""
"""ERR3211766""","""AtH2023_ERR1135319_MAG02_1_23""",1878,878,0.467519,0.643237,1.375854,"""s__Bariatricus sp004560705"""
"""ERR3211766""","""AtH2023_ERR1135319_MAG02_1_27""",2493,1415,0.567589,1.186923,2.091166,"""s__Bariatricus sp004560705"""
…,…,…,…,…,…,…,…
"""SRR11125751""","""AtH2023_SRR8960831_MAG08_7_11""",126,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595"""
"""SRR11125751""","""AtH2023_ERR1135443_MAG08_149_1""",504,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595"""
"""SRR11125751""","""AtH2023_SRR14369154_MAG04_52_1…",2283,29,0.012703,0.012703,1.0,"""s__UBA2868 sp004552595"""


In [6]:
depth_df.filter(pl.col('depth_all').is_not_nan()).sort(by='depth_all', descending=True).filter(pl.col('depth_all') > 0.0)

metag,gene,len,hits,breadth,depth_all,depth_cov,species
str,str,u32,u32,f64,f64,f64,str
"""SRR17241521""","""AtH2023_SRR11183406_MAG1_196_4…",324,324,1.0,16832.611111,16832.611111,"""s__Lactobacillus amylovorus"""
"""SRR10209683""","""AtH2023_SRR8960486_MAG03_96_2""",375,375,1.0,15132.666667,15132.666667,"""s__Mogibacterium_A kristiansen…"
"""SRR11489783""","""AtH2023_SRR8960486_MAG03_96_2""",375,375,1.0,11149.749333,11149.749333,"""s__Mogibacterium_A kristiansen…"
"""ERR8314769""","""AtH2023_SRR8960486_MAG03_96_2""",375,375,1.0,11009.386667,11009.386667,"""s__Mogibacterium_A kristiansen…"
"""ERR8314733""","""AtH2023_SRR8960486_MAG03_96_2""",375,375,1.0,10793.218667,10793.218667,"""s__Mogibacterium_A kristiansen…"
…,…,…,…,…,…,…,…
"""SRR11183784""","""AtH2023_SRR14369225_MAG28_16_7…",3246,1,0.000308,0.000616,2.0,"""s__Cryptobacteroides sp0004349…"
"""SRR8960444""","""AtH2023_ERR1135259_MAG16_32_5""",3480,2,0.000575,0.000575,1.0,"""s__UBA2868 sp004552595"""
"""SRR8960492""","""AtH2023_ERR3211862_MAG2_81_1""",4197,2,0.000477,0.000477,1.0,"""s__Prevotella sp002251295"""


In [16]:
#depth_df.write_csv('../mapping-depth.csv')

## Summarize our mapping breadth results across all the metagenomes

This is for figuring out which genes to manually curate.

In [ ]:
assert 0

In [ ]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [ ]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    # get fraction of columns where breadth is greater than cutoff as 'f'
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f"),
#    (pl.col("depth").filter(pl.col("depth").is_not_nan()).mean()),
)
agg_df

In [ ]:
# print information out by species
for species in sorted(agg_df['species'].unique()):
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    foo_df = foo_df.sort(by='f', descending=True).filter(pl.col('f') > 0.8)
    print(foo_df)

    top50_names = set(foo_df.head(50)['gene'].to_list())

    outfile = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.fa'
    print(outfile)
    outfp = open(outfile, 'wt')
    for record in screed.open(f'../outputs.cds/singleclust/{species}.cds3.min50.dedup.fa'):
        name = record.name.split(' ')[0]
        if name in top50_names:
            top50_names.remove(name)
            outfp.write(f'>{record.name}\n{record.sequence}\n')
    assert not top50_names
    outfp.close()

    outfile2 = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.csv'
    print(outfile2)
    outfp = open(outfile2, 'w', newline='')
    w = csv.writer(outfp)

    for name in foo_df.head(50)['gene'].to_list():
        w.writerow(['0', '0', species, name, "(not reviewed)"])
    outfp.close()
            
    